# YOLOv8s training on LS-SSDD (Kaggle, resumable)

**What we get in Output** (`/kaggle/working` as we used in Kaggle):
- `runs/yolov8s_lsssdd/weights/best.pt`: the model to use for Sentinel-1 inference
- `runs/yolov8s_lsssdd/weights/resume.pt`: the full checkpoint for resuming
- `runs/yolov8s_lsssdd/epoch_vis/`: one picture per epoch of the same val tile, plus `training.gif`
- `runs/yolov8s_lsssdd/results.png`: loss and mAP curves
- `test_metrics.txt`: final numbers on the test split (scenes 11 to 15)

Training is resumable and information for each epoch is stored in `resume.pt`.

In [ ]:
!pip install -q ultralytics

## Settings

In [ ]:
import os, shutil, time, glob
from pathlib import Path
import numpy as np, cv2, matplotlib.pyplot as plt
import torch, yaml
from ultralytics import YOLO

INPUT_ROOT = Path("/kaggle/input")
WORK       = Path("/kaggle/working")
RUN_NAME   = "yolov8s_lsssdd"

# Model. "yolov8s.pt" = standard YOLOv8s pretrained on COCO (start here).
# "yolov8s-p2.yaml" = YOLOv8s with an extra high-resolution head for tiny objects (experiment 2).
MODEL = "yolov8s.pt"

IMGSZ, EPOCHS, BATCH, PATIENCE = 800, 100, 16, 30   # use BATCH=8 for the P2 model
TIME_BUDGET_HOURS = 11.0     # stop cleanly before Kaggle's 12 h limit, leaving time for final eval
VIS_CONF = 0.25              # confidence threshold for the per-epoch picture

# Learning-rate strategy (explained in the next markdown cell)
HYP = dict(optimizer="SGD", lr0=0.01, lrf=0.01, momentum=0.937, weight_decay=5e-4,
           warmup_epochs=3, cos_lr=True, close_mosaic=10)
# Augmentation: same as previewed in the preprocessing notebook
AUG = dict(hsv_h=0.0, hsv_s=0.0, hsv_v=0.2, degrees=0.0, translate=0.1, scale=0.5,
           fliplr=0.5, flipud=0.5, mosaic=1.0, mixup=0.0)

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (CPU, will be very slow)")

### Learning-rate strategy
- **SGD, starting LR 0.01, momentum 0.937.** This is YOLOv8's standard recipe for fine-tuning from COCO weights. SGD is set explicitly, because YOLO's `optimizer="auto"` picks a different optimizer depending on dataset size, which makes runs harder to compare.
- **Warmup for 3 epochs.** The LR climbs from near zero to 0.01. This stops the first, noisy updates from wrecking the pretrained weights.
- **Cosine decay down to 0.01 × 0.01 = 0.0001.** The LR drops smoothly. It stays high longer than with a straight-line decay and flattens near the end, which lets the model fine-tune the exact box edges. Tiny ships need that.
- **Mosaic off for the last 10 epochs** (`close_mosaic`). The model finishes on whole, natural-looking tiles like the ones it will see at inference.
- **Early stop after 30 epochs with no gain** on val mAP50-95 (`patience`).

## Find the dataset and any checkpoint to resume from

In [ ]:
# Dataset: the folder from the preprocessing notebook that has data.yaml + images/train
cands = [p.parent for p in INPUT_ROOT.rglob("data.yaml") if (p.parent / "images" / "train").is_dir()]
assert cands, "Preprocessed dataset not found. Add the preprocessing notebook's Output as an Input."
DATA_DIR = cands[0]
DATA_YAML = WORK / "data.yaml"
DATA_YAML.write_text(f"path: {DATA_DIR}\ntrain: images/train\nval: images/val\ntest: images/test\n\nnc: 1\nnames: ['ship']\n")
for split in ("train", "val", "test"):
    print(f"{split:5s}: {len(list((DATA_DIR / 'images' / split).glob('*')))} images")

# Resume: a resume.pt from a previous version of THIS notebook, attached as an Input
RUN_DIR = WORK / "runs" / RUN_NAME
RESUME_CKPT = None
prev = sorted(INPUT_ROOT.rglob(f"{RUN_NAME}/weights/resume.pt"), key=os.path.getmtime)
if prev and not (RUN_DIR / "weights" / "resume.pt").exists():
    src = prev[-1].parent.parent
    ck = torch.load(prev[-1], map_location="cpu", weights_only=False)
    done, total = ck.get("epoch", -1) + 1, ck["train_args"]["epochs"]
    print(f"found previous run at {src}: {done}/{total} epochs done")
    if 0 < done < total:
        shutil.copytree(src, RUN_DIR, dirs_exist_ok=True)      # bring back logs, curves, epoch pictures
        RESUME_CKPT = RUN_DIR / "weights" / "resume.pt"
    del ck
elif (RUN_DIR / "weights" / "resume.pt").exists():
    RESUME_CKPT = RUN_DIR / "weights" / "resume.pt"            # same session, re-running the cell
print("resume from:", RESUME_CKPT or "nothing, starting a new run")

## Pick the fixed validation tile
The val tile with the most ships. The same tile is drawn after every epoch: **green = ground truth, red = model prediction** (with confidence). Early on, expect no red boxes or random ones. As training goes on, the red boxes should land on the green ones.

In [ ]:
def read_labels(lbl_path, w, h):
    boxes = []
    if lbl_path.exists():
        for ln in lbl_path.read_text().split("\n"):
            if ln.strip():
                _, cx, cy, bw, bh = map(float, ln.split())
                boxes.append(((cx-bw/2)*w, (cy-bh/2)*h, (cx+bw/2)*w, (cy+bh/2)*h))
    return boxes

val_lbls = list((DATA_DIR / "labels" / "val").glob("*.txt"))
VIS_LBL = max(val_lbls, key=lambda p: len([l for l in p.read_text().split("\n") if l.strip()]))
VIS_IMG = next((DATA_DIR / "images" / "val").glob(VIS_LBL.stem + ".*"))
vis_im = cv2.imread(str(VIS_IMG))
VIS_GT = read_labels(VIS_LBL, vis_im.shape[1], vis_im.shape[0])
print(f"fixed val tile: {VIS_IMG.name} with {len(VIS_GT)} ships")

VIS_DIR = RUN_DIR / "epoch_vis"

def draw_epoch(model, title, out_path):
    pred = model.predict(str(VIS_IMG), imgsz=IMGSZ, conf=VIS_CONF, verbose=False)[0]
    im = cv2.cvtColor(vis_im, cv2.COLOR_BGR2RGB)
    for x1, y1, x2, y2 in VIS_GT:
        cv2.rectangle(im, (int(x1)-2, int(y1)-2), (int(x2)+2, int(y2)+2), (0, 255, 0), 1)
    for (x1, y1, x2, y2), c in zip(pred.boxes.xyxy.cpu().numpy(), pred.boxes.conf.cpu().numpy()):
        cv2.rectangle(im, (int(x1), int(y1)), (int(x2), int(y2)), (255, 0, 0), 2)
        cv2.putText(im, f"{c:.2f}", (int(x1), max(int(y1) - 3, 10)), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255, 0, 0), 1)
    fig = plt.figure(figsize=(7.5, 7.8))
    plt.imshow(im); plt.axis("off")
    plt.title(f"{title}\nGT {len(VIS_GT)} ships (green) | predicted {len(pred.boxes)} (red, conf>={VIS_CONF})", fontsize=10)
    plt.tight_layout(); fig.savefig(out_path, dpi=100); plt.show(); plt.close(fig)

draw_epoch(YOLO(MODEL if MODEL.endswith(".pt") else "yolov8s.pt"), "before training (COCO weights)",
           WORK / "before_training.png")

## Callbacks: per-epoch picture, full checkpoint, time limit
- **Per-epoch picture:** after each epoch, the latest weights predict on the fixed tile.
- **`resume.pt`:** at the end of training YOLO strips the optimizer from `last.pt`, which makes it impossible to resume. So after every epoch the notebook also keeps a full copy as `resume.pt`.
- **Time limit:** once `TIME_BUDGET_HOURS` has passed, training stops cleanly after the current epoch.

In [ ]:
T0 = time.time()

def on_model_save(trainer):
    shutil.copy2(trainer.last, trainer.wdir / "resume.pt")

_in_epoch = False   # True between a training epoch's end and its fit-epoch-end; False during YOLO's final eval

def on_train_epoch_end(trainer):
    global _in_epoch
    _in_epoch = True

def on_fit_epoch_end(trainer):
    global _in_epoch
    VIS_DIR.mkdir(parents=True, exist_ok=True)
    m = trainer.metrics or {}
    scores = (f"val mAP50 {m.get('metrics/mAP50(B)', 0):.3f}  mAP50-95 {m.get('metrics/mAP50-95(B)', 0):.3f}  "
              f"P {m.get('metrics/precision(B)', 0):.2f}  R {m.get('metrics/recall(B)', 0):.2f}")
    if _in_epoch:   # normal end of an epoch: draw with the latest weights
        _in_epoch = False
        draw_epoch(YOLO(str(trainer.last)), f"epoch {trainer.epoch + 1}/{trainer.epochs}  |  {scores}",
                   VIS_DIR / f"epoch_{trainer.epoch + 1:03d}.png")
        if (time.time() - T0) / 3600 > TIME_BUDGET_HOURS and not trainer.stop:
            print(f"\nTime budget of {TIME_BUDGET_HOURS} h reached. Stopping after epoch {trainer.epoch + 1}; "
                  f"resume.pt is saved. Attach this notebook's Output as Input and commit again to continue.")
            trainer.stop = True
    elif trainer.best.exists():   # YOLO's final evaluation of best.pt
        draw_epoch(YOLO(str(trainer.best)), f"final: best.pt  |  {scores}", VIS_DIR / "final_best.png")

## Train

In [ ]:
if RESUME_CKPT:
    model = YOLO(str(RESUME_CKPT))
else:
    model = YOLO(MODEL) if MODEL.endswith(".pt") else YOLO(MODEL).load("yolov8s.pt")   # P2: copy matching COCO weights
model.add_callback("on_model_save", on_model_save)
model.add_callback("on_train_epoch_end", on_train_epoch_end)
model.add_callback("on_fit_epoch_end", on_fit_epoch_end)

common = dict(device=0 if torch.cuda.is_available() else "cpu", workers=4, plots=True)
if RESUME_CKPT:
    model.train(resume=True, data=str(DATA_YAML), save_dir=str(RUN_DIR), **common)
else:
    model.train(data=str(DATA_YAML), imgsz=IMGSZ, epochs=EPOCHS, batch=BATCH, patience=PATIENCE,
                save_period=-1, seed=0, project=str(WORK / "runs"), name=RUN_NAME, exist_ok=True,
                **HYP, **AUG, **common)

## Results

In [ ]:
from IPython.display import Image as IPImage, display
from PIL import Image

if (RUN_DIR / "results.png").exists():
    display(IPImage(str(RUN_DIR / "results.png"), width=1000))

frames = sorted(VIS_DIR.glob("epoch_*.png"))
if frames:
    ims = [Image.open(f).convert("P", palette=Image.ADAPTIVE) for f in frames]
    ims[0].save(VIS_DIR / "training.gif", save_all=True, append_images=ims[1:], duration=400, loop=0)
    print(f"saved {VIS_DIR / 'training.gif'} ({len(frames)} frames)")

best = RUN_DIR / "weights" / "best.pt"
if best.exists():
    r = YOLO(str(best)).val(data=str(DATA_YAML), split="test", imgsz=IMGSZ, batch=BATCH,
                            project=str(WORK / "runs"), name="test_eval", exist_ok=True)
    txt = (f"best.pt on TEST (scenes 11-15)\nprecision {r.box.mp:.4f}\nrecall    {r.box.mr:.4f}\n"
           f"mAP50     {r.box.map50:.4f}\nmAP50-95  {r.box.map:.4f}\n")
    (WORK / "test_metrics.txt").write_text(txt); print(txt)